# Visualization

可视化数据文件

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
PATH_SLOT = ""
PARTICIPANT_ID = 0

In [ ]:
MODE = "formal" # "review"表示看待审核数据，"formal"则是从已接受的正式数据中抽取某一被试
PRACTICE_ITEMS = ["肥皂"]  # 练习物品名称

if MODE == "review":
    FILE_PATH = f"../data/to_review/创意用途生成任务{PATH_SLOT}实验_2.csv"
elif MODE == "formal":
    # 从participants.csv中获取被试ID对应的文件名
    participant_data = pd.read_csv("../data/preprocessed/participants.csv")
    FILE_NAME = participant_data[participant_data['participant_id'] == PARTICIPANT_ID]['psychopy_file'].values[0]
    FILE_PATH = f"../data/raw/psychopy/{FILE_NAME}"


## 事件时间线

In [ ]:
def get_y_jitter(key_times):
    # return 0.3 + np.random.uniform(-0.1, 0.1, size=len(key_times))    # 添加随机抖动，避免点重叠
    return np.full(len(key_times), 0.3)  # 固定在0.3位置

In [ ]:
# 读取 CSV，自动处理 BOM
df = pd.read_csv(FILE_PATH, encoding='utf-8-sig')

# 筛选正式物品行：itemName 非空且不在练习列表中
item_rows = df[df['itemName'].notna() & ~df['itemName'].isin(PRACTICE_ITEMS)]

# 准备绘图
fig, axes = plt.subplots(len(item_rows), 1, figsize=(14, 8), sharex=False)
if len(item_rows) == 1:
    axes = [axes]

for ax, (idx, row) in zip(axes, item_rows.iterrows()):
    item_name = row['itemName']
    messages_str = row.get('messages', '[]')
    try:
        messages = json.loads(messages_str) if pd.notna(messages_str) else []
    except json.JSONDecodeError:
        messages = []


    # 解析 KeyRespRecord.rt 字段（被试按键时间）
    key_str = row.get('KeyRespRecord.rt', '[]')
    try:
        # 字段可能是字符串形式的列表，如 "[8.11, 8.27, ...]"，也可能为空
        key_times = json.loads(key_str) if pd.notna(key_str) and key_str.strip() else []
        # key_times = [key_times[0], key_times[-1]]
    except (json.JSONDecodeError, TypeError):
        key_times = []
    # 确保元素都是数值，并过滤可能的 None
    key_times = [float(t) for t in key_times if t is not None]

    # 获取 AUT 开始和结束时间（可能为 NaN，此时根据消息时间范围设定）
    try:
        start_time = float(row['AUT.started'])
        stop_time = float(row['AUT.stopped'])

        # 提取被试和AI的提交时间（相对于任务开始）
        user_times = [msg['time'] - start_time for msg in messages if msg.get('role') == 'user']
        assistant_times = [msg['time'] - start_time for msg in messages if msg.get('role') == 'assistant']
    except (ValueError, TypeError, KeyError):
        all_times = user_times + assistant_times
        start_time = min(all_times) - 5 if all_times else 0
        stop_time = max(all_times) + 5 if all_times else 100

    # 绘制事件线
    ax.eventplot(user_times, orientation='horizontal', colors='steelblue',
                 lineoffsets=0, linelengths=0.4, label='被试')
    ax.eventplot(assistant_times, orientation='horizontal', colors='orange',
                 lineoffsets=0, linelengths=0.4, label='AI')
    if key_times:
        y_jitter = get_y_jitter(key_times)
        ax.scatter(key_times, y_jitter, color='green',
               marker='o', s=10, label='按键反应', zorder=5)

    # 添加时间标签（用户在上方，助手在下方）
    # for t in user_times:
    #     ax.text(t, 0.15, f'{t:.1f}s', ha='center', va='bottom', fontsize=7, rotation=45, color='steelblue')
    # for t in assistant_times:
    #     ax.text(t, -0.15, f'{t:.1f}s', ha='center', va='top', fontsize=7, rotation=45, color='orange')

    # 设置轴范围与格式
    ax.set_xlim(0, 300)
    ax.set_ylim(-0.5, 0.5)

    ax.xaxis.set_major_locator(MultipleLocator(30))

    ax.set_yticks([])
    ax.set_xlabel('时间 (秒)')
    ax.set_title(f'{item_name}')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 按键反应时间分析

In [ ]:
# 检查相邻按键间隔（单位：秒）
all_dt = []
per_item = []

for _, row in item_rows.iterrows():
    item_name = row["itemName"]
    key_str = row.get("KeyRespRecord.rt", "[]")
    try:
        key_times = json.loads(key_str) if pd.notna(key_str) and str(key_str).strip() else []
    except Exception:
        key_times = []
    key_times = sorted([float(t) for t in key_times if t is not None])

    if len(key_times) >= 2:
        dt = np.diff(key_times)
        all_dt.extend(dt.tolist())
        per_item.append({
            "item": item_name,
            "n_keys": len(key_times),
            "min_dt_ms": dt.min() * 1000,
            "p10_dt_ms": np.percentile(dt, 10) * 1000,
            "median_dt_ms": np.median(dt) * 1000,
            "dt_lt_50ms": int((dt < 0.05).sum()),
            "dt_lt_20ms": int((dt < 0.02).sum()),
        })
    else:
        per_item.append({
            "item": item_name,
            "n_keys": len(key_times),
            "min_dt_ms": np.nan,
            "p10_dt_ms": np.nan,
            "median_dt_ms": np.nan,
            "dt_lt_50ms": 0,
            "dt_lt_20ms": 0,
        })

summary_df = pd.DataFrame(per_item)
display(summary_df.sort_values(["dt_lt_20ms", "dt_lt_50ms"], ascending=False))

if all_dt:
    all_dt = np.array(all_dt)
    print(f"全局中位间隔: {np.median(all_dt)*1000:.1f} ms")
    print(f"<50ms 占比: {(all_dt < 0.05).mean()*100:.2f}%")
    print(f"<20ms 占比: {(all_dt < 0.02).mean()*100:.2f}%")

## 回答详情

In [ ]:
from IPython.display import display, HTML

# 收集每个物品的消息DataFrame（使用相对时间）及InputField内容
item_dfs = []
for idx, row in item_rows.iterrows():
    item_name = row['itemName']
    start_time = float(row['AUT.started'])  # 任务开始时间
    # 获取InputField文本
    input_text = row.get('InputField.text', '')
    if pd.isna(input_text):
        input_text = ''  # 转为空字符串
    
    messages_str = row.get('messages', '[]')
    try:
        messages = json.loads(messages_str) if pd.notna(messages_str) else []
    except json.JSONDecodeError:
        messages = []
    if messages:
        records = []
        for msg in messages:
            rel_time = msg['time'] - start_time
            records.append({
                'role': msg.get('role'),
                'content': msg.get('content'),
                'time': round(rel_time, 2)
            })
        df_item = pd.DataFrame(records).sort_values('time').reset_index(drop=True)
    else:
        df_item = None
    # 存储物品名称、消息DataFrame、输入文本
    item_dfs.append((item_name, df_item, input_text))

# 每两个一组并排显示
for i in range(0, len(item_dfs), 2):
    pair = item_dfs[i:i+2]
    html_parts = []
    for name, df, input_txt in pair:
        # 回答数量
        if df is not None:
            user_answer_num = (df['role'] == 'user').sum()
            assistant_answer_num = (df['role'] == 'assistant').sum()
        else:
            user_answer_num = 0
            assistant_answer_num = 0
        
        # 构建单个物品的HTML：标题 + 消息表格 + 输入框内容
        if df is not None:
            table_html = df.to_html(index=True, escape=False)
        else:
            table_html = "<p>无消息</p>"
        # 输入框内容显示（如果存在）
        if input_txt and input_txt.strip():
            input_display = f'<div style="margin-top:12px;"><strong>输入框内容：</strong><br><pre style="white-space: pre-wrap; margin-top:4px;">{input_txt}</pre></div>'
        else:
            input_display = '<div style="margin-top:12px;">无输入框内容</div>'
        
        html_parts.append(f"""
        <div style="flex: 1; margin-right: 10px; min-width: 300px;">
            <h4>{name} ({user_answer_num} + {assistant_answer_num})</h4>
            {table_html}
            {input_display}
        </div>
        """)
    # 用flex容器包裹并排放置
    combined_html = f'<div style="display: flex; flex-wrap: wrap; gap: 10px;">{"".join(html_parts)}</div>'
    display(HTML(combined_html))

## 按键统计

In [ ]:
from IPython.display import display, HTML

# 收集每个物品的按键统计
item_key_dfs = []
for idx, row in item_rows.iterrows():
    item_name = row['itemName']
    keys_str = row.get('KeyRespRecord.keys', '[]')
    try:
        keys = json.loads(keys_str) if pd.notna(keys_str) and keys_str.strip() else []
    except (json.JSONDecodeError, TypeError):
        keys = []
    # 转换为字符串，处理 None（PsychoPy 中可能出现的空值）
    keys = [str(k) if k is not None else 'None' for k in keys]
    
    if keys:
        # 统计频次
        key_counts = pd.Series(keys).value_counts().reset_index()
        key_counts.columns = ['按键', '次数']
        # 添加总计行
        total_row = pd.DataFrame({'按键': ['总计'], '次数': [len(keys)]})
        key_counts = pd.concat([key_counts, total_row], ignore_index=True)
    else:
        key_counts = pd.DataFrame({'按键': ['无按键'], '次数': [0]})
    
    item_key_dfs.append((item_name, key_counts))

# 并排显示每个物品的按键分布
html_parts = []
for name, df in item_key_dfs:
    table_html = df.to_html(index=False, escape=False)
    html_parts.append(f"""
    <div style="flex: 1 1; margin-right: 10px; margin-bottom: 10px">
        <h4>{name}</h4>
        {table_html}
    </div>
    """)

combined_html = f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">{"".join(html_parts)}</div>'
display(HTML(combined_html))